## Train Neural Network Parameterisation

Train deterministic neural network, inputs = X, outputs = U. We will also validate and test the model (offline)

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset, DataLoader


from ml_models.TorchModels import  NN
from plotting_scripts.plot_data_histogram import plot_hist




ModuleNotFoundError: No module named 'plotting_scripts'

In [ ]:
# Random seed
seed = 123
np.random.seed(seed)

# Define dimensions of system (fixed)
K = 8   
J = 32  

# Define the "true" parameters
h = 1
F = 20
c = 10
b = 10

# Define time-stepping
dt = 0.001        # Two layer (high-res) run at dt
dt_f = 0.005      # One layer (coarse-res) run at dt_f




In [3]:
# Data path should already exist as we have already run the notebook to generate training data
data_path = f'./outputs/'
# Get data
X = np.load(f'{data_path}/X_train_dtf.npy')
U = np.load(f'{data_path}/U_train_dtf.npy')
print(f'Data loaded from {data_path}')
print(X.shape)

Data loaded from ./outputs/
(199999, 8)


In [12]:
X = X[::10]
U = U[::10]

In [7]:
X[::10].shape

(20000, 8)

Training parameters

In [16]:
N_train = 10000
BATCH_SIZE = 128
NUM_EPOCHS = 20


In [17]:

N = X.shape[0]
N_val = max(N - N_train, 0)   # Use remainder for validation
print(N_val)

features = np.ravel(X[:N_train])   
targets = np.ravel(U[:N_train])   

features_val = np.ravel(X[N_train:])   
targets_val = np.ravel(U[N_train:])    

print(features.shape, targets.shape, features_val.shape, targets_val.shape)


10000
(80000,) (80000,) (80000,) (80000,)


In [18]:

X_torch = torch.tensor(features, dtype=torch.float32).reshape((-1, 1))
Y_torch = torch.tensor(targets, dtype=torch.float32).reshape((-1, 1))

X_val = torch.tensor(features_val, dtype=torch.float32).reshape((-1, 1))
Y_val = torch.tensor(targets_val, dtype=torch.float32).reshape((-1, 1))


dataset = TensorDataset(X_torch, Y_torch)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

dataset_val = TensorDataset(X_val, Y_val)
dataloader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=True)


In [25]:
# Set up neural network model
nn_model = NN(n_features=1, n_targets=1, n_hidden=[16, 16], activation='ReLU')

In [ ]:
# Optimisation settings
optimiser = torch.optim.Adam(params = nn_model.parameters(), lr=1e-2)
loss_function = torch.nn.MSELoss()


NameError: name 'model' is not defined

In [26]:
losses_all = []
losses_train = []
losses_val = []
min_loss = 1E8

for epoch in range(NUM_EPOCHS):
    loss_train = 0.
    for X_batch, Y_batch in dataloader:
        nn_model.train()
        optimiser.zero_grad()
        pred = nn_model(X_batch)
        loss = loss_function(pred, Y_batch)
        loss.backward()

        losses_all.append(loss.item())
        loss_train += loss.item()/X_batch.shape[0]

        # Update optimiser
        optimiser.step()
    losses_train.append(loss_train)
    loss_val = 0.
    for X_batch, Y_batch in data_loader_val:
        # validation
        nn_model.eval()
        pred = nn_model(X_batch)
        loss = loss_function(pred, Y_batch)
        losses_val += loss.item()/X_batch.shape[0]

    losses_val.append(loss_val)
    if loss_val < min_loss:
        # Save checkpoint
        output_dicts = {
            "iteration": iteration,
            "val_loss": losses_val[-1],
            "train_loss": losses[-1],
            "model": nn_model,
            "np_rng_state": np.random.get_state(),
            "torch_rng_state": torch.random.get_rng_state()
            }

        torch.save(output_dicts, f"{data_path}/deterministic_nn_model.pt")
        min_loss = loss_val

    
    print("Done training")


NameError: name 'optimiser' is not defined

In [ ]:

# Plot losses
plt.clf()
figure, ax = plt.subplots(1)
plt.semilogy(losses, color="k")
plt.semilogy(losses_val, color="b")
plt.xlabel("Iterations")
plt.ylabel("MSE Loss")


In [ ]:

    # Plot and save result
    nn_model.eval()
    plt.clf()
    
    # Plot
    Xmin=-9
    Xmax=16
    Ymin=-21
    Ymax=16
    figure, (ax1, ax2) = plt.subplots(2, 1, sharex=True, 
        gridspec_kw={'height_ratios': [5, 1], 'hspace':0})
    X_domain = torch.linspace(Xmin, Xmax, 80).unsqueeze(-1)

    # Plot raw data
    if predict_sigma:
        Y_torch = Y_torch[:, 0]
    plt.sca(ax1)
    plt.scatter(X_torch.flatten()[::], Y_torch.flatten()[::], color="k", alpha=0.2, label="Training data")
    plt.axis(ymin=Ymin, ymax=Ymax, xmin=Xmin, xmax=Xmax)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.xlabel("$X$", fontsize=18)
    plt.ylabel("$U$", fontsize=18)
    plt.tight_layout()
    plt.savefig(f"{save_model_path}/data.png")
    print(f"{save_model_path}/data.png")

    # Deterministic prediction 
    out = nn_model(X_domain).detach()
    if out.shape[1] > 1:
        det_pred, det_sigma = out.chunk(2, dim=-1)
        det_sigma = torch.exp(det_sigma)
    else:
        det_pred = out

    plt.plot(X_domain.squeeze(), det_pred, color="k", linewidth=2, label="Deterministic NN")
    if out.shape[1] > 1:
        plt.plot(X_domain.squeeze(), det_pred+det_sigma, color="k", linestyle="dashed", linewidth=2, label="output 2")
        plt.plot(X_domain.squeeze(), det_pred-det_sigma, color="k", linestyle="dashed", linewidth=2, label="output 2")

    plt.legend()
    # Add histogram
    plt.sca(ax2)
    plot_hist(X_torch.flatten()[::], ax2)
    plt.axis(ymin=0, xmin=Xmin, xmax=Xmax)
    plt.yticks([], fontsize=18)
    plt.xticks(fontsize=18)
    plt.xlabel("$X$", fontsize=18)
    plt.tight_layout()
    plt.savefig(f"{save_model_path}/{save_prefix}offline_deterministic.png")
    print(f"{save_model_path}/{save_prefix}offline_deterministic.png")
    plt.close()

